In [19]:
import random, time, numpy as np, torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.FashionMNIST(
    root='data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(
    root='data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False)

In [ ]:
class FashionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # STUDENT: define three trainable Linear layers
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784,128)
        self.fc2 = nn.Linear(128,64)
        self.fc3 = nn.Linear(64,10)
        self.activation = nn.ReLU()

    def forward(self, x):
        # STUDENT: flatten, apply Layer 1 + activation,
        # Layer 2 + activation, then Layer 3 logits
        x = self.flatten(x)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        logits = self.fc3(x)

        return logits

model = FashionMLP().to(device)
print(model)
print()

total_params = sum(p.numel() for p in model.parameters())
print("Total Trainable Parameters:", total_params)
print()

images, labels = next(iter(train_loader))
images = images.to(device)
print("Input Shape:", images.shape)
print()

x = model.flatten(images)
print("After Flatten:", x.shape)

x = model.activation(model.fc1(x))
print("After Layer 1:", x.shape)

x = model.activation(model.fc2(x))
print("After Layer 2:", x.shape)

x = model.fc3(x)
print("After Layer 3:", x.shape)


FashionMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
  (activation): ReLU()
)

Total Trainable Parameters: 109386

Input Shape: torch.Size([64, 1, 28, 28])

After Flatten: torch.Size([64, 784])
After Layer 1: torch.Size([64, 128])
After Layer 2: torch.Size([64, 64])
After Layer 3: torch.Size([64, 10])


Layer 1 uses 784 inputs because the picture has 28x28 pixels, which is a 2 dimensional matrix and it needs to be flattened into a one-dimensional vector. Since it has 28x28 pixels, the total is 784, so the first layer requires 784 input neurons. ReLU was use because this helps the model focus on importamt signals and makes training faster. Layer 3 has 10 outputs because Fashion-MNIST has 10 classes. Each output neuron produces one logit corresponding to one class. Logits are basically raw scores, and when the model returns logits, the loss function will use them to check how wrong the prediction is by comparing it with the correct answer.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train_one_epoch(model, loader):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()          # clear previous gradients
        logits = model(images)         # forward propagation
        loss = criterion(logits, labels)
        loss.backward()                # backpropagation
        optimizer.step()               # parameter update

        running_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


In [ ]:
#Evaluation Function
def evaluate(model, loader):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)

            loss = criterion(logits, labels)

            running_loss += loss.item() * labels.size(0)

            correct += (logits.argmax(1) == labels).sum().item()

            total += labels.size(0)

    return running_loss / total, correct / total

In [ ]:
from sklearn.metrics import f1_score

def macro_f1(model, loader):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            logits = model(images)

            preds = logits.argmax(1).cpu().numpy()

            all_preds.extend(preds)

            all_labels.extend(labels.numpy())

    score = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return score

In [ ]:
import time
import copy

def run_experiment(model, optimizer, epochs, run_name, scheduler=None):
    criterion = nn.CrossEntropyLoss()
    total_params = sum(p.numel() for p in model.parameters())

    best_acc = 0.0
    best_state = None

    start_time = time.time()
    for epoch in range(epochs):
        train_loss, _ = train_one_epoch(model, train_loader)
        test_loss, test_accuracy = evaluate(model, test_loader)
        if scheduler is not None:
            scheduler.step()
        if test_accuracy > best_acc:
            best_acc = test_accuracy
            best_state = copy.deepcopy(model.state_dict())
    training_time = time.time() - start_time

    model.load_state_dict(best_state)

    start_time = time.time()
    test_loss, test_accuracy = evaluate(model, test_loader)
    inference_time = time.time() - start_time
    f1 = macro_f1(model, test_loader)

    print(f"{run_name} Results")
    print()
    print("Total Parameters:", total_params)
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Macro F1 Score: {f1:.4f}")
    print(f"Training Time: {training_time:.2f} seconds")
    print(f"Inference Time: {inference_time:.4f} seconds")
    return model

In [ ]:
class FashionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.activation = nn.ReLU()
    def forward(self, x):
        x = self.flatten(x)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        return self.fc3(x)

model = FashionMLP().to(device)
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
model = run_experiment(model, optimizer, epochs=30, run_name="Run A", scheduler=scheduler)

Run A Results

Total Parameters: 109386
Train Loss: 0.1339
Test Loss: 0.3163
Test Accuracy: 0.8931
Macro F1 Score: 0.8931
Training Time: 619.49 seconds
Inference Time: 2.1453 seconds


In [ ]:
model = FashionMLP().to(device)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
model = run_experiment(model, optimizer, epochs=30, run_name="Run B", scheduler=scheduler)

Run B Results

Total Parameters: 109386
Train Loss: 0.1206
Test Loss: 0.3339
Test Accuracy: 0.8962
Macro F1 Score: 0.8953
Training Time: 689.84 seconds
Inference Time: 2.2261 seconds


In [ ]:
class FashionMLP256(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 10)
        self.activation = nn.ReLU()
    def forward(self, x):
        x = self.flatten(x)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        return self.fc3(x)

model = FashionMLP256().to(device)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
model = run_experiment(model, optimizer, epochs=30, run_name="Run C", scheduler=scheduler)

Run C Results

Total Parameters: 218058
Train Loss: 0.0963
Test Loss: 0.3375
Test Accuracy: 0.8995
Macro F1 Score: 0.8992
Training Time: 832.33 seconds
Inference Time: 4.0349 seconds


In [ ]:
class FashionMLPTanh(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.activation = nn.Tanh()
    def forward(self, x):
        x = self.flatten(x)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        return self.fc3(x)

model = FashionMLPTanh().to(device)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
model = run_experiment(model, optimizer, epochs=30, run_name="Run D", scheduler=scheduler)

Run D Results

Total Parameters: 109386
Train Loss: 0.1171
Test Loss: 0.3267
Test Accuracy: 0.8906
Macro F1 Score: 0.8906
Training Time: 641.03 seconds
Inference Time: 2.4215 seconds
